In [1]:
!pip install requests beautifulsoup4 pandas

In [2]:
# Step 1: Import libraries and set up configuration
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-IN,en;q=0.9"
}

BASE_URL = "https://www.amazon.in/s"

In [3]:
# Step 2: Function to get search page HTML
def get_search_page(page=1):
    params = {
        "k": "laptop",
        "page": page
    }
    resp = requests.get(BASE_URL, params=params, headers=HEADERS)
    resp.raise_for_status()
    return resp.text

print("get_search_page function defined")

get_search_page function defined


In [4]:
# Step 3: Function to parse products from HTML
def parse_products(html):
    soup = BeautifulSoup(html, "html.parser")
    results = []

    # Each product container
    for item in soup.select("div.s-main-slot div[data-component-type='s-search-result']"):
        # Title
        title_el = item.select_one("h2 a span")
        title = title_el.get_text(strip=True) if title_el else None

        # Image URL
        img_el = item.select_one("img.s-image")
        image_url = img_el["src"] if img_el and img_el.has_attr("src") else None

        # Rating
        rating_el = item.select_one("span.a-icon-alt")
        rating = rating_el.get_text(strip=True) if rating_el else None

        # Price
        price_whole = item.select_one("span.a-price-whole")
        price_frac = item.select_one("span.a-price-fraction")
        if price_whole:
            price = price_whole.get_text(strip=True).replace(",", "")
            if price_frac:
                price = f"{price}.{price_frac.get_text(strip=True)}"
        else:
            price = None

        # Ad / Organic (look for "Sponsored" label)
        is_ad = bool(item.select_one("span[aria-label='Sponsored']") or
                     item.select_one("span:contains('Sponsored')"))

        # Only add if we have at least a title
        if title:
            results.append({
                "image_url": image_url,
                "title": title,
                "rating": rating,
                "price": price,
                "ad_or_organic": "Ad" if is_ad else "Organic"
            })

    return results

print("parse_products function defined")

parse_products function defined


In [5]:
# Step 4: Main function to scrape and save data
def main():
    all_products = []

    # Scrape first 2 pages
    for page in range(1, 3):
        print(f"Scraping page {page}...")
        html = get_search_page(page)
        products = parse_products(html)
        all_products.extend(products)
        print(f"Found {len(products)} products on page {page}")
        time.sleep(1)

    df = pd.DataFrame(all_products)

    # Timestamped filename: laptops_YYYYmmdd_HHMMSS.csv
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = f"laptops_{timestamp}.csv"

    print(f"\nSaving {len(df)} records to {output_path}")
    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("\n--- Scraping Complete! ---")
    print(f"Total products scraped: {len(df)}")
    print(f"CSV file saved: {output_path}")

    # Display first five records
    print("\nFirst 5 records:")
    print(df.head())

    return df

print("main function defined")

main function defined


In [6]:
# Step 5: Run the scraper!
print("Starting Amazon Laptop Scraper...\n")
df_result = main()

Starting Amazon Laptop Scraper...

Scraping page 1...


/usr/local/lib/python3.12/dist-packages/soupsieve/css_parser.py:876: FutureWarning: The pseudo class ':contains' is deprecated, ':-soup-contains' should be used moving forward.
  warnings.warn(  # noqa: B028


Found 0 products on page 1
Scraping page 2...
Found 0 products on page 2

Saving 0 records to laptops_20251114_180356.csv

--- Scraping Complete! ---
Total products scraped: 0
CSV file saved: laptops_20251114_180356.csv

First 5 records:
Empty DataFrame
Columns: []
Index: []


In [7]:
#this section is to check what is the mistake
html_sample = get_search_page(1)
print(f"HTML Length: {len(html_sample)}")
print("\nFirst 1000 characters:")
print(html_sample[:1000])

# Check for product containers
soup = BeautifulSoup(html_sample, "html.parser")
product_containers = soup.select("div[data-component-type='s-search-result']")
print(f"\nFound {len(product_containers)} product containers")

HTML Length: 1291007

First 1000 characters:
<!doctype html><html lang="en-in" class="a-no-js" data-19ax5a9jf="dingo"><!-- sp:feature:head-start -->
<head><script>var aPageStart = (new Date()).getTime();</script><meta charset="utf-8"/>
<!-- sp:end-feature:head-start -->
<!-- sp:feature:csm:head-open-part1 -->

<script type='text/javascript'>var ue_t0=ue_t0||+new Date();</script>
<!-- sp:end-feature:csm:head-open-part1 -->
<!-- sp:feature:cs-optimization -->
<meta http-equiv='x-dns-prefetch-control' content='on'>
<link rel="dns-prefetch" href="https://images-eu.ssl-images-amazon.com">
<link rel="dns-prefetch" href="https://m.media-amazon.com">
<link rel="dns-prefetch" href="https://completion.amazon.com">
<!-- sp:end-feature:cs-optimization -->
<!-- sp:feature:csm:head-open-part2 -->
<script type='text/javascript'>
window.ue_ihb = (window.ue_ihb || window.ueinit || 0) + 1;
if (window.ue_ihb === 1) {

var ue_csm = window,
    ue_hob = +new Date();
(function(d){var e=d.ue=d.ue||{},f=Date.

In [8]:
# modified parser - we use more flexible selectors
def parse_products_v2(html):
    soup = BeautifulSoup(html, "html.parser")
    results = []

    # Each product container
    for item in soup.select("div[data-component-type='s-search-result']"):
        # try multiple selectors
        title = None
        title_el = item.select_one("h2 span") or item.select_one("h2") or item.select_one("[data-cy='title-recipe']")
        if title_el:
            title = title_el.get_text(strip=True)

        # Image URL
        img_el = item.select_one("img")
        image_url = None
        if img_el:
            image_url = img_el.get("src") or img_el.get("data-src")

        # Rating
        rating_el = item.select_one("span.a-icon-alt") or item.select_one("i.a-icon-star")
        rating = rating_el.get_text(strip=True) if rating_el else None

        # Price - try multiple approaches
        price = None
        price_whole = item.select_one("span.a-price-whole")
        if price_whole:
            price_text = price_whole.get_text(strip=True).replace(",", "").replace(".", "")
            price_frac = item.select_one("span.a-price-fraction")
            if price_frac:
                price = f"{price_text}.{price_frac.get_text(strip=True)}"
            else:
                price = price_text
        else:
            # Alternative price selector
            price_span = item.select_one("span.a-price span.a-offscreen")
            if price_span:
                price = price_span.get_text(strip=True)

        # Ad / Organic
        is_ad = bool(item.select_one("[data-component-type='sp-sponsored-result']") or
                     item.find(string=lambda text: text and 'Sponsored' in text))

        # Only add if we have at least a title
        if title:
            results.append({
                "image_url": image_url,
                "title": title,
                "rating": rating,
                "price": price,
                "ad_or_organic": "Ad" if is_ad else "Organic"
            })

    return results

print("Improved parser defined")

Improved parser defined


In [9]:
# Testing the modified parser
test_results = parse_products_v2(html_sample)
print(f"Found {len(test_results)} products with improved parser")
if test_results:
    print("\nFirst product:")
    print(test_results[0])

Found 16 products with improved parser

First product:
{'image_url': 'https://m.media-amazon.com/images/I/712cUkgrVnL._AC_UY218_.jpg', 'title': 'HP 255 G10 Laptop for Home or Work, 16GB RAM, 512GB SSD, 15.6" Full HD, Ryzen 3 7330U (Beats Intel i5-1135G7), HDMI, USB-C, Windows 11 Pro, Business and Fun Ready', 'rating': '4.6 out of 5 stars', 'price': '32540', 'ad_or_organic': 'Organic'}


In [10]:
# Run the full scraper with modified parser
print("=" * 50)
print("Running Amazon Laptop Scraper with Improved Parser")
print("=" * 50)

all_products = []

# Scrape first 2 pages
for page in range(1, 3):
    print(f"\nScraping page {page}...")
    html = get_search_page(page)
    products = parse_products_v2(html)  # Using improved parser
    all_products.extend(products)
    print(f"Found {len(products)} products on page {page}")
    time.sleep(1)  # Be polite

df = pd.DataFrame(all_products)

# Timestamped filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = f"laptops_{timestamp}.csv"

print(f"\n" + "="*50)
print(f"Saving {len(df)} records to {output_path}")
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\n--- Scraping Complete! ---")
print(f"Total products scraped: {len(df)}")
print(f"CSV file saved: {output_path}")
print("\nFirst 5 records:")
print(df.head())
print("\nColumn summary:")
print(df.info())

Running Amazon Laptop Scraper with Improved Parser

Scraping page 1...
Found 16 products on page 1

Scraping page 2...
Found 16 products on page 2

Saving 32 records to laptops_20251114_180745.csv

--- Scraping Complete! ---
Total products scraped: 32
CSV file saved: laptops_20251114_180745.csv

First 5 records:
                                           image_url  \
0  https://m.media-amazon.com/images/I/712cUkgrVn...   
1  https://m.media-amazon.com/images/I/61Y8j1QN5Y...   
2  https://m.media-amazon.com/images/I/71h+7L8gcr...   
3  https://m.media-amazon.com/images/I/61AccNkmFF...   
4  https://m.media-amazon.com/images/I/61Kfqt-h5-...   

                                               title              rating  \
0  HP 255 G10 Laptop for Home or Work, 16GB RAM, ...  4.6 out of 5 stars   
1  Acer Aspire 3, Intel Celeron N4500, 12 GB RAM,...  3.4 out of 5 stars   
2  BrowseBook 14.1" FHD IPS Laptop | Best Student...  3.2 out of 5 stars   
3  Lenovo V15 G4 AMD Athlon Silver 7120U Lapt

In [12]:
# Download the CSV file
from google.colab import files

print("\n" + "="*50)
print("Download the CSV file")
print("="*50)

# Download the file
files.download(output_path)
print(f"\nFile {output_path} is ready for download!")
print(f"\nTotal laptops scraped: {len(df)}")
print(f"Columns: {', '.join(df.columns.tolist())}")

print("\nTask Complete!\n")


Download the CSV file


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


File laptops_20251114_180745.csv is ready for download!

Total laptops scraped: 32
Columns: image_url, title, rating, price, ad_or_organic

Task Complete!

